In [ ]:
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
from tqdm import tqdm

from src.data.dataloader import get_dataloader
from src.models.mm_encoder import MultiModalEncoder

In [ ]:
def retriev_ts_embeddings(args, model, data_loader):
    model.eval()
    all_ts_embeddings = []
    all_text_embeddings = []
    all_timeseries = []
    all_sample_ids = []

    with torch.no_grad():
        for batch_x in tqdm(data_loader, total=len(data_loader)):
            timeseries = batch_x.timeseries.float().to(args.device)
            input_mask = batch_x.input_mask.long().to(args.device)
            sample_ids = torch.tensor(batch_x.sample_id, dtype=torch.long).reshape(-1).to(args.device)

            outputs = model(
                x_enc=timeseries,
                input_mask=input_mask,
                channel_description_emb=batch_x.channel_description_emb.to(args.device),
                description_emb=batch_x.description_emb.to(args.device),
                event_emb=batch_x.event_emb.to(args.device),
            )

            all_timeseries.append(timeseries)
            all_ts_embeddings.append(outputs.embeddings)
            all_text_embeddings.append(outputs.description_emb)
            all_sample_ids.append(sample_ids)

    return {
        "timeseries": torch.cat(all_timeseries, dim=0),
        "ts_embeddings": F.normalize(torch.cat(all_ts_embeddings, dim=0), dim=-1),
        "text_embeddings": F.normalize(torch.cat(all_text_embeddings, dim=0), dim=-1),
        "sample_ids": torch.cat(all_sample_ids, dim=0),
    }

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def load_and_embed(checkpoint_path, split="train"):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    args = checkpoint["args"]

    model = MultiModalEncoder(args)
    state_dict = {k.replace("module.", ""): v for k, v in checkpoint["model_state_dict"].items()}
    model.load_state_dict(state_dict)
    model.to(device)

    args.task_name = "retrieval"
    args.data_split = split
    args.batch_size = args.train_batch_size
    args.device = device
    args.distributed = False

    data_loader = get_dataloader(args)
    output = retriev_ts_embeddings(args, model, data_loader)

    del model
    torch.cuda.empty_cache()
    return output

In [ ]:
checkpoint_path = "results/model_checkpoints/context_align/retriever_demo.pt"
out = load_and_embed(checkpoint_path=checkpoint_path, split="train")

print("ts_embeddings:", tuple(out["ts_embeddings"].shape))
print("text_embeddings:", tuple(out["text_embeddings"].shape))
print("timeseries:", tuple(out["timeseries"].shape))

In [ ]:
# Optional: save embeddings for later reuse
output_dir = Path("results/embeddings")
output_dir.mkdir(parents=True, exist_ok=True)

np.save(output_dir / "ts_embedding.npy", out["ts_embeddings"].cpu().numpy())
np.save(output_dir / "text_embedding.npy", out["text_embeddings"].cpu().numpy())
np.save(output_dir / "timeseries.npy", out["timeseries"].cpu().numpy())
np.save(output_dir / "sample_ids.npy", out["sample_ids"].cpu().numpy())

In [ ]:
parquet_path = Path("dataset/retrieval/train.parquet")
df = pq.read_table(parquet_path).to_pandas()

query_idx = 0  # change this index
query_text_embedding = out["text_embeddings"][query_idx : query_idx + 1]
all_ts_embeddings = out["ts_embeddings"]

mask = torch.arange(all_ts_embeddings.size(0), device=all_ts_embeddings.device) != query_idx
candidate_scores = torch.mm(query_text_embedding, all_ts_embeddings[mask].T).squeeze(0)

topk = 5
topk_scores, topk_pos = torch.topk(candidate_scores, k=topk)
candidate_indices = torch.where(mask)[0][topk_pos]

print("query sample_id:", int(out["sample_ids"][query_idx]))
print("top-k candidate indices:", candidate_indices.tolist())

In [ ]:
query_id = int(out["sample_ids"][query_idx])
query_row = df[df["id"] == query_id].iloc[0]

print("\n[Query]")
print("id:", query_id)
print("description:", query_row["description"])
print("events:", query_row["events"])

print("\n[Retrieved Top-K]")
for rank, (idx, score) in enumerate(zip(candidate_indices.tolist(), topk_scores.tolist()), start=1):
    sample_id = int(out["sample_ids"][idx])
    row = df[df["id"] == sample_id].iloc[0]
    print(f"Top-{rank} | id={sample_id} | score={score:.4f}")
    print("  description:", row["description"])
    print("  events:", row["events"])
